# AUX 01 — SarcOji Validation of the Frozen Sarcasm-Indicated Proxy

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Use SarcOji only as auxiliary validation/limitation evidence for the already frozen surface-cue proxy; never for sentiment model selection or threshold tuning.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

This notebook intentionally does **not** modify any TweetEval subgroup rule. It should be run only after `FINAL 00` and the SarcOji source file is available. The existing frozen EDA-05 notebook remains the methodological reference; this notebook provides a compact final-run wrapper.

In [4]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

import re, pandas as pd, numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN


In [5]:
# Reproducibly obtain the official SarcOji source used by EDA_05.
# This avoids depending on a manually copied SARCOJI.df in My Drive.
import requests, hashlib

REPO_URL = "https://github.com/VanditaGroverKapila/SarcOji"
RAW_URL = "https://raw.githubusercontent.com/VanditaGroverKapila/SarcOji/main/SARCOJI.df"
SRC = Path("/content/SARCOJI.df")

r = requests.get(RAW_URL, timeout=120)
r.raise_for_status()
SRC.write_bytes(r.content)
sha256 = hashlib.sha256(r.content).hexdigest()

# SarcOji is distributed as a pandas pickle/DataFrame object.
work = pd.read_pickle(SRC)

# Repository-level integrity checks carried forward from EDA_05.
assert len(work) == 29377, f"Unexpected SarcOji row count: {len(work):,}"
lower_cols = {str(c).lower(): c for c in work.columns}
assert "text" in lower_cols and "sarcastic" in lower_cols, \
    f"Expected Text/Sarcastic columns; found: {work.columns.tolist()}"

print("✅ Official SarcOji source downloaded")
print("Source:", RAW_URL)
print("SHA256:", sha256)
print("Shape:", work.shape)
print("Columns:", work.columns.tolist())


✅ Official SarcOji source downloaded
Source: https://raw.githubusercontent.com/VanditaGroverKapila/SarcOji/main/SARCOJI.df
SHA256: acdc7b40db84ca7e076f32f06eb74d861c86d5e0ac7752db1b13d4df93d35d2e
Shape: (29377, 16)
Columns: ['Sarcastic', 'Text', 'Emojis', 'MaxEmoji', 'MaxEmojiNumOccurence', 'MaxEmojiPos', 'TextSWN', 'TextVader', 'TextTextBlob', 'EmojiSWN', 'EmojiVader', 'EmojiTextBlob', 'MEmojiWN', 'MEVader', 'METB', 'ESR']


In [6]:
lower={str(c).lower():c for c in work.columns}
text_col=lower.get("text")
label_col=lower.get("sarcastic") or lower.get("label")
assert text_col and label_col
x=work.copy()
x["text"]=x[text_col].fillna("").astype(str)
x["gold"]=pd.to_numeric(x[label_col],errors="raise").astype(int)
assert set(x.gold.unique()).issubset({0,1})
tag=tag_subgroups(x,"text")
proxy=(tag["subgroup"]=="sarcasm-indicated").astype(int).to_numpy()
gold=x.gold.to_numpy()

cm=confusion_matrix(gold,proxy,labels=[0,1])
metrics={
    "N":len(x),
    "Gold Sarcastic":int((gold==1).sum()),
    "Proxy Positive":int(proxy.sum()),
    "Precision":precision_score(gold,proxy,zero_division=0),
    "Recall":recall_score(gold,proxy,zero_division=0),
    "F1":f1_score(gold,proxy,zero_division=0),
    "Balanced Accuracy":balanced_accuracy_score(gold,proxy),
    "Rule Changed After Validation":False,
    "Permitted Role":"auxiliary validation/limitation evidence only",
}
pd.DataFrame([metrics]).to_csv(FINAL["results"]/"AUX01_SarcOji_proxy_validation.csv",index=False)
print(metrics)
print(cm)

{'N': 29377, 'Gold Sarcastic': 11448, 'Proxy Positive': 63, 'Precision': 0.4126984126984127, 'Recall': 0.0022711390635918936, 'F1': 0.004517418121796542, 'Balanced Accuracy': np.float64(0.5001037216875214), 'Rule Changed After Validation': False, 'Permitted Role': 'auxiliary validation/limitation evidence only'}
[[17892    37]
 [11422    26]]


### Interpretation guard
A weak or strong SarcOji score does not trigger rule tuning. SarcOji is emoji-restricted and cannot establish general social-media sarcasm prevalence, authorial intent, or a gold sarcasm label for TweetEval.